# Customer Churn Prediction using Logistic Regression
Predicting customer churn for a telecommunications company based on demographic details and service usage.

---

## Task 1: Data Understanding
- Load the dataset using Pandas
- Inspect initial records
- Identify numerical features, categorical features, and the target variable

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

In [ ]:
# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan))

# Identify feature types and target variable
target = 'Churn'
features = [col for col in df.columns if col not in ['customerID', target]]

num_cols = df[features].select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df[features].select_dtypes(include=['object']).columns.tolist()

print('Target variable:', target)
print('Numerical features:', num_cols)
print('Categorical features:', cat_cols)

--- 
## Task 2: Data Preprocessing
- Check and handle missing values
- Encode categorical variables
- Split data into 80% training and 20% testing sets

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill missing TotalCharges with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

In [ ]:
# Encode target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# One-hot encode categorical features
df_encoded = pd.get_dummies(df.drop(columns=['customerID']), drop_first=True)

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

--- 
## Task 3: Model Development
Train a Logistic Regression model to predict customer churn.

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predictions on test set
y_pred = model.predict(X_test_scaled)

--- 
## Task 4: Model Evaluation
Evaluate model performance using accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1-Score : {f1:.4f}\n')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Observations:
1. The model achieves an overall accuracy around 80.5%.
2. Precision (~66%) is higher than recall (~54%), meaning predictions for positive churners are generally accurate, but some actual churners are missed.
3. Class imbalance (around 73% non-churn vs 27% churn) shifts the model slightly toward predicting non-churn.

--- 
## Task 5: Conclusion

In this project, a Logistic Regression model was built to predict customer churn using telecom subscription data. The analysis shows that customer churn is strongly linked to contract duration and tenure length, with month-to-month contract holders exhibiting the highest risk of churn.

The model achieved an accuracy of 80.5% and a precision of 66%. A main limitation of Logistic Regression here is its assumption of a linear decision boundary, which prevents it from naturally modeling complex feature interactions without manual feature engineering. Class imbalance also lowers recall for churned customers. Future work could test tree-based models like Random Forests along with sampling techniques like SMOTE.